# Baselines e preparação das dobras do projeto de dengue

Notebook do Google Colab para executar os baselines **B0, B1 e B2** e preparar as dobras LORO persistidas, usadas depois pelo treinamento M1 em [`04_modelos.ipynb`](./04_modelos.ipynb).

- B0: prevalência do desfecho no conjunto de treino.
- B1: regressão logística com perfil de comorbidades.
- B2: regra clínica de referência do protocolo.

O notebook usa as funções do pacote `src/dengue_sinan/`; não contém lógica de modelagem. Execute-o primeiro para gerar `m1/dobras.npz`, o relatório `reports/verificacao/T-2.3.json` e os baselines OOF no Google Drive. O notebook M1 não carrega nem treina com 2024.

**Privacidade:** não publique microdados nem artefatos OOF. As saídas deste notebook são métricas agregadas; os artefatos ficam no Google Drive montado na sua conta.

## 1. Preparar o ambiente

O projeto exige `Python >=3.13,<3.14`. O branch `ricardo` precisa estar publicado no GitHub e conter os arquivos listados abaixo. Se o branch ainda não existir remotamente, publique-o ou use o branch que recebeu o merge antes de continuar.

In [ ]:
import sys

if not ((3, 13) <= sys.version_info[:2] < (3, 14)):
    raise RuntimeError(
        f"Python {sys.version_info.major}.{sys.version_info.minor} detectado. "
        "Este projeto exige Python >=3.13,<3.14. Selecione um runtime compatível e reinicie."
    )

print(sys.version)

In [ ]:
import subprocess
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

REPO = Path("/content/dengue-comorbidity-risk-ricardo")
REPOSITORY = "https://github.com/PabloThePmX/dengue-comorbidity-risk.git"
BRANCH = "ricardo"
remote_branch = subprocess.run(
    ["git", "ls-remote", "--heads", REPOSITORY, f"refs/heads/{BRANCH}"],
    check=True,
    capture_output=True,
    text=True,
)
if not remote_branch.stdout.strip():
    raise RuntimeError(
        f"O branch {BRANCH!r} ainda não está publicado no GitHub. "
        "Publique as alterações ou altere BRANCH para o branch já atualizado."
    )
if not REPO.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, REPOSITORY, str(REPO)],
        check=True,
    )
else:
    current_branch = subprocess.run(
        ["git", "-C", str(REPO), "branch", "--show-current"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    if current_branch != BRANCH:
        raise RuntimeError(
            f"O clone existente está em {current_branch!r}, não em {BRANCH!r}. "
            "Reinicie o runtime do Colab para clonar o branch correto."
        )
    subprocess.run(
        ["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH],
        check=True,
    )

required_files = [
    REPO / "config" / "base.yaml",
    REPO / "config" / "esperado.yaml",
    REPO / "src" / "dengue_sinan" / "baselines.py",
    REPO / "src" / "dengue_sinan" / "dataset.py",
    REPO / "src" / "dengue_sinan" / "folds.py",
    REPO / "src" / "dengue_sinan" / "runner.py",
]
missing_files = [str(path) for path in required_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError(
        "O clone não contém todos os arquivos necessários. Atualize/publice o repositório "
        "e execute novamente. Arquivos ausentes: " + ", ".join(missing_files)
    )

%cd /content/dengue-comorbidity-risk-ricardo
%pip install -e .

## 2. Configurar dados e saídas

Os dados devem estar no Google Drive nos Parquets anuais criados pelo notebook de profiling. Se os arquivos não existirem, interrompa aqui e execute [`profiling/01_profiling.ipynb`](../profiling/01_profiling.ipynb) primeiro.

In [ ]:
from dengue_sinan.config import load_config

PROJECT_DIR = Path("/content/drive/MyDrive/dengue-comorbidity-risk")
DATA_DIR = PROJECT_DIR / "data"
REPORTS_DIR = PROJECT_DIR / "reports"
MLRUNS_DIR = PROJECT_DIR / "mlruns"
MODELS_DIR = PROJECT_DIR / "models"
M1_DIR = PROJECT_DIR / "m1"

for directory in (DATA_DIR, REPORTS_DIR, MLRUNS_DIR, MODELS_DIR, M1_DIR):
    directory.mkdir(parents=True, exist_ok=True)

config = load_config(REPO / "config" / "base.yaml")
config["caminhos"]["dados"] = str(DATA_DIR)
config["caminhos"]["relatorios"] = str(REPORTS_DIR)
config["caminhos"]["mlruns"] = str(MLRUNS_DIR)
config["caminhos"]["modelos"] = str(MODELS_DIR)
config["caminhos"]["m1"] = str(M1_DIR)
config["caminhos"]["dobras"] = str(M1_DIR / "dobras.npz")
config["caminhos"]["esperado"] = str(REPO / "config" / "esperado.yaml")

print(f"Dados: {DATA_DIR}")
print(f"Relatórios: {REPORTS_DIR}")
print(f"Experimentos MLflow: {MLRUNS_DIR}")
print(f"Dobras e progresso M1: {M1_DIR}")

In [ ]:
import json

ANOS_DESENVOLVIMENTO = [2022, 2023]
manifest_path = DATA_DIR / "manifest.json"
if not manifest_path.is_file():
    raise FileNotFoundError(
        "Manifesto de dados ausente no Drive. Execute primeiro "
        "profiling/01_profiling.ipynb para baixar e converter os arquivos."
    )

manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
required_data_files = []
for ano in ANOS_DESENVOLVIMENTO:
    entrada = manifest.get(str(ano))
    if not isinstance(entrada, dict) or not {"parquet", "arquivo"}.issubset(entrada):
        raise ValueError(f"Manifesto sem metadados completos para {ano}.")
    required_data_files.extend(
        [DATA_DIR / entrada["parquet"], DATA_DIR / "raw" / entrada["arquivo"]]
    )

missing_data_files = [str(path) for path in required_data_files if not path.is_file()]
if missing_data_files:
    raise FileNotFoundError(
        "Arquivos de desenvolvimento ausentes no Drive. Execute primeiro "
        "profiling/01_profiling.ipynb. Ausentes: " + ", ".join(missing_data_files)
    )

print("Manifesto e dados de 2022 e 2023 encontrados. O ano de teste (2024) não será carregado.")

## 3. Carregar somente os dados de desenvolvimento

O conjunto de teste de 2024 permanece intocado. A função do pacote aplica os filtros da população, deriva o alvo e as features e grava um fluxograma agregado.

In [ ]:
from dengue_sinan.dataset import load_population
from dengue_sinan.folds import verify_and_save_folds

df_dev = load_population(config, anos=ANOS_DESENVOLVIMENTO)

anos_carregados = sorted(df_dev["ANO"].dropna().astype(int).unique().tolist())
if anos_carregados != ANOS_DESENVOLVIMENTO:
    raise RuntimeError(
        f"Anos inesperados carregados: {anos_carregados}; "
        f"esperados: {ANOS_DESENVOLVIMENTO}."
    )
if 2024 in set(df_dev["ANO"].astype(int)):
    raise RuntimeError("O conjunto de teste de 2024 não pode entrar no desenvolvimento.")

resumo_dev = (
    df_dev.groupby("ANO", observed=True)
    .agg(casos=("y", "size"), eventos=("y", "sum"))
    .assign(prevalencia=lambda tabela: tabela["eventos"] / tabela["casos"])
)
display(resumo_dev)
print(f"Total agregado: {len(df_dev):,} casos; {int(df_dev['y'].sum()):,} eventos.")

expected = load_config(REPO / "config" / "esperado.yaml")
fold_path, fold_hash, fold_checks = verify_and_save_folds(
    df_dev,
    M1_DIR / "dobras.npz",
    REPORTS_DIR,
    expected["eventos_por_regiao"],
    config["ordem_dobras"],
)
if not all(check["passou"] for check in fold_checks):
    raise RuntimeError(f"A verificação das dobras falhou: {fold_checks}")
print(f"Dobras salvas e verificadas: {fold_path}; hash {fold_hash}")

## 4. Executar os baselines

A função `run_and_log_baselines` executa as dobras regionais definidas no projeto, grava métricas agregadas e registra os artefatos OOF no MLflow persistido no Drive. Não exiba nem compartilhe arquivos OOF: eles contêm uma linha por caso.

In [ ]:
from dengue_sinan.baselines import run_and_log_baselines

metricas, _oof = run_and_log_baselines(df_dev, config)
print("Baselines concluídos.")

In [ ]:
colunas_agregadas = [
    coluna
    for coluna in (
        "modelo", "regiao", "pr_auc", "brier", "roc_auc", "acuracia", "precisao",
        "recall", "f1", "f2"
    )
    if coluna in metricas.columns
]
display(metricas[colunas_agregadas])

relatorio = REPORTS_DIR / "baselines.md"
print(f"Relatório agregado: {relatorio}")
print(f"MLflow persistido no Drive: {MLRUNS_DIR}")

## 5. Encerrar

Após confirmar que os relatórios agregados foram gravados, use **Ambiente de execução → Desconectar e excluir ambiente de execução** para liberar recursos. Não compartilhe o Drive, o diretório `mlruns` ou artefatos OOF com dados linha a linha.

Para treinar M1, execute [`04_modelos.ipynb`](./04_modelos.ipynb) depois deste notebook. O teste final de 2024 não faz parte deste fluxo.